# 04 锁

## 八股问答

### 1. MySQL 中有哪些锁？

按粒度：

| 锁 | 范围 | 典型场景 |
| --- | --- | --- |
| 全局锁 | 整个实例 | 全库备份时 `FLUSH TABLES WITH READ LOCK` |
| 表级锁 | 整张表 | DDL、显式 `LOCK TABLES`、MyISAM 写 |
| 行级锁 | 具体行 | InnoDB 默认，冲突最小 |

按模式：共享锁 S（读锁，可共存）和排他锁 X（写锁，互斥）。S 和 S 兼容，
S 与 X、X 与 X 都互斥。

### 2. 全局锁是什么？

全局锁是对整个 MySQL 实例加锁。执行 `FLUSH TABLES WITH READ LOCK`
（FTWRL）后，整个实例进入只读状态：`SELECT` 仍可以执行，但写数据的
DML（`INSERT`、`UPDATE`、`DELETE`）、DDL，以及已更新数据事务的
`COMMIT` 都会被阻塞。执行 `UNLOCK TABLES` 或断开持锁会话后释放全局锁。

![全局锁保证全库备份一致性](assets/global-lock-backup-consistency.png)

典型场景是全库逻辑备份。如果备份期间业务仍在写入，可能出现库存表
已备份、订单表尚未备份时业务又同时扣减库存并生成订单，最终得到
的多张表不属于同一时点。全局锁通过暂停写入，让备份看到一致的
数据状态。对仅使用 InnoDB 的库，通常优先使用
`mysqldump --single-transaction` 基于 MVCC 获取一致性快照，避免长时间阻塞业务写入。

### 3. 表级锁有哪些？

MySQL 的表级锁主要分为表锁、元数据锁（MDL）、意向锁和自增锁
（AUTO-INC）。

- 表锁：通过 `LOCK TABLES` 显式加锁，又分为表共享读锁和表独占写锁。

| 表锁类型 | 加锁语句 | 持锁会话 | 其他会话 |
| --- | --- | --- | --- |
| 表共享读锁 | `LOCK TABLES table_name READ` | 可读、不可写 | 可读，写入被阻塞 |
| 表独占写锁 | `LOCK TABLES table_name WRITE` | 可读、可写 | 读写都被阻塞 |

InnoDB 日常业务很少显式使用表锁；MyISAM 主要使用表级锁控制并发。
- 元数据锁 MDL：执行 DML 时自动获取，DDL 会等 DML 结束；长事务会让 DDL
  一直排队，这是线上“加字段卡住”的常见原因。
- 意向锁 IS/IX：InnoDB 行锁之上的表级标记。事务要给行加 S 锁，先给表加 IS；
  加 X 锁，先加 IX。作用是在表级快速判断“这张表有没有行锁冲突”，避免逐行
  检查。
- 自增锁 AUTO-INC：插入自增列时为保证值不重复加的锁，`innodb_autoinc_lock_mode`
  控制是语句级还是插入级加锁。

### 4. 行级锁有哪些？

- 记录锁 Record Lock：锁单条索引记录。
- 间隙锁 Gap Lock：锁两条索引记录之间的区间，只阻止插入，不锁已有记录。
- 临键锁 Next-Key Lock：记录锁 + 前一个间隙，锁左开右闭区间，是 InnoDB RR
  下默认的行锁，同时防幻读。
- 插入意向锁 Insert Intention Lock：插入前对间隙的“预约”标记，多个插入
  意向锁互相兼容，但和间隙锁冲突。

补充：行锁必须挂在索引上。如果更新条件没有可用索引，InnoDB 会锁住所有扫描
到的行，退化成表级效果，这也是“更新一定要走索引”的原因之一。

### 5. 记录锁 + 间隙锁能防止删除导致的幻读吗？

能。幻读定义是两次范围查询结果行数不同，删除也是制造幻读的途径之一。

- 防插入：间隙锁锁住区间，其他事务无法在区间内插入。
- 防删除：记录锁锁住已存在的记录，其他事务无法删除它们。
- 临键锁同时包含两者：锁“记录及前面间隙”，RR 下范围当前读不会让行数变化。

注意：如果只在 RR 下用快照读，靠的是 MVCC 而不是锁；锁是给当前读用的。

### 6. 死锁是什么？怎么办？

死锁是多个事务互相持有对方需要的锁，形成循环等待。产生条件：互斥、持有并
等待、不可剥夺、循环等待。

InnoDB 的应对：

1. 死锁检测：检测到循环等待后，回滚代价较小的事务，报 `1213 Deadlock found`。
2. 锁等待超时：`innodb_lock_wait_timeout`（默认 50 秒）到期放弃，报 `1205`。

排查命令：

```sql
SHOW ENGINE INNODB STATUS\G
SELECT * FROM information_schema.innodb_trx;
SELECT * FROM information_schema.innodb_lock_waits;
```

预防手段：按固定顺序加锁、尽量缩小事务范围、避免长事务、控制单事务更新行数、
把热点行操作串行化。


## 面试话术

“先分粒度：全局锁用于备份，表锁里有 MDL 和意向锁，InnoDB 日常冲突主要发生在
行锁。RR 下默认临键锁 = 记录锁 + 间隙锁，防幻读的同时也扩大了阻塞范围，所以
高并发场景可以评估换 RC 减少间隙锁。死锁我遇到过，处理方式是先看
innodb_trx 和 lock_waits 找出互相等待的两条链路，然后统一加锁顺序、缩短事务，
并把 innodb_lock_wait_timeout 调到一个业务可接受的阈值。”


## 自查清单

- [ ] 能说清全局锁、表级锁、行级锁的分类和用途
- [ ] 能说清记录锁、间隙锁、临键锁、插入意向锁的区别
- [ ] 能解释为什么记录锁 + 间隙锁能防删除导致的幻读
- [ ] 能在双终端复现行锁阻塞、间隙锁拦截和死锁
- [ ] 能通过 `SHOW ENGINE INNODB STATUS` 找到死锁的两个事务
